In [0]:
from pyspark.sql.functions import *

data = [
    ("Hello World",),
    ("Databricks is amazing",),
    ("Spark makes big data fast",)
]


df = spark.createDataFrame(data, ["message"])

display(df)

In [0]:
df_lower = df.withColumn("message", lower("message"))
display(df_lower)

In [0]:
df_replace = df_lower.withColumn("vowels", regexp_replace("message", "a|e|i|o|u|\s", ""))
display(df_replace)

In [0]:
df_count = df_replace.withColumn("count", length("vowels"))
display(df_count.select("vowels","count"))

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

data = [
    ('A001', 'Tokyo', 'Seoul'),
    ('A001', 'Singapore', 'Tokyo'),
    ('A001', 'Seoul', 'Bangkok'),
    ('A001', 'Bangkok', 'Manila'),
    ('B002', 'Paris', 'Rome'),
    ('B002', 'Berlin', 'Paris'),
    ('B002', 'Rome', 'Madrid'),
    ('C003', 'Cairo', 'Dubai'),
    ('C003', 'Istanbul', 'Cairo'),
    ('C003', 'Dubai', 'Riyadh'),
    ('C003', 'Riyadh', 'Doha'),
    ('D004', 'Toronto', 'Montreal'),
    ('D004', 'Vancouver', 'Toronto'),
    ('E005', 'Sydney', 'Melbourne')
]

schema = StructType([
    StructField("customer", StringType(), True),
    StructField("start_location", StringType(), True),
    StructField("end_location", StringType(), True)
])

df = spark.createDataFrame(data=data, schema=schema)

display(df)

In [0]:
df_start = df.select("customer","start_location")\
           .subtract(df.select("customer",col("end_location").alias("start_location")))


df_end = df.select("customer","end_location")\
    .subtract(df.select("customer",col("start_location").alias("end_location")))

df_join = df_start.join(df_end,on = "customer",how = "inner")

display(df_join)

In [0]:
data = [
    (1,"Alice", 1200, "Q1"),
    (2,"Bob", 900, "Q1"),
    (3,"Charlie", 1500, "Q2"),
    (4,"David", 1700, "Q2"),
    (5,"Eva", 1100, "Q3"),
    (6,"Frank", 220, "Q3"),
    (7,"Grace", 1300, "Q4")
]

df = spark.createDataFrame(data, ["id", "name", "sales", "quarter"])
df.display()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

df_new_col = df.withColumn("next", lead("name").over(Window.orderBy("id")))\
    .withColumn("previous", lag("name").over(Window.orderBy("id")))

df_interchange = df_new_col.withColumn("inter", when(col("id") % 2 != 0, coalesce(col("next"),col("name")))\
    .when(col("id") % 2 == 0, col("previous")))

display(df_interchange.select("id","name","inter"))



In [0]:
data = [1,4,5,7,9,10]
schema = ["id"]

df = spark.createDataFrame(data = data, schema = schema)
display(df)

In [0]:
from pyspark.sql.functions import *

df_range = df.agg(min(col("id")).alias("min_val"),max(col("id")).alias("max_val")).collect()[0]

#row = df.agg(min(col("id")).alias("min_id"),max(col("id")).alias("max_id")).collect()[0]

min = df_range["min_val"]
max = df_range["max_val"]

df_new_range = spark.range(min,max+1)
#display(df_new_range)

df_join = df_new_range.join(df,"id","left_anti")
display(df_join)


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import StructType, StructField, StringType



# Define schema
schema = StructType([
    StructField("show_date", StringType(), True),
    StructField("show_status", StringType(), True)
])

# Data
data = [
    ("01/06/20", "Booked"),
    ("02/06/20", "Booked"),
    ("03/06/20", "Booked"),
    ("04/06/20", "Available"),
    ("05/06/20", "Available"),
    ("06/06/20", "Available"),
    ("07/06/20", "Booked")
]

# Create DataFrame
df = spark.createDataFrame(data, schema=schema)

# Show DataFrame
df.show()

In [0]:
display(df)

In [0]:
from pyspark.sql.window import Window

Window_spec = Window.orderBy("show_date")
df_lag = df.withColumn("prev_show", lag("show_status").over(Window_spec))\
    .withColumn("status_flag", when(col("show_status") != col("prev_show"),1).otherwise(0))\
        .withColumn("grp",sum(col("status_flag")).over(Window.orderBy("show_date")))\
            .groupBy("grp","show_status").agg(min("show_date").alias("s_date"),max("show_date").alias("e_date"))\
                .drop("grp")
display(df_lag)

In [0]:
from pyspark.sql.functions import *


data = [
    (1, "Alice", None, 10000),
    (2, "Bob", 1, 8000),
    (3, "Charlie", 1, 7000),
    (4, "David", 2, 9000),
    (5, "Eva", 2, 6000),
    (6, "Frank", 3, 5000),
    (7, "Grace", 3, 12000)
]

columns = ["EmployeeID", "EmployeeName", "ManagerID", "Salary"]

df = spark.createDataFrame(data, columns)

In [0]:
display(df)

In [0]:
df_join = df.alias("emp").join(df.alias("manager"),col("emp.ManagerID") == col("manager.EmployeeID"),"inner")
df_join.select(col("emp.EmployeeID").alias("EmployeeID"),col("emp.EmployeeName").alias("EmployeeName"),col("emp.Salary").alias("Salary"),col("manager.EmployeeName").alias("ManagerName"),col("manager.Salary").alias("Managersalary")).filter(col("Salary") > col("Managersalary")).display()


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
spark = SparkSession.builder.getOrCreate()

data = [
    (1, "Alice", "HR", 5000),
    (2, "Bob", "HR", 6000),
    (3, "Charlie", "IT", 7000),
    (4, "David", "IT", 9000),
    (5, "Eva", "Finance", 8000),
    (6, "Frank", "Finance", 5000),
    (7, "Grace", "Finance", 12000),
    (8, "Lorence", "IT", 12000)
]

columns = ["EmployeeID", "EmployeeName", "Department", "Salary"]

df = spark.createDataFrame(data, columns)
df_3 = df.withColumn("avg_sal", avg(col("Salary")).over(Window.partitionBy("Department")))
display(df_3.filter(col("Salary")>col("avg_sal")))

In [0]:
from pyspark.sql.functions import *


data = [
    ("Forest", ["sddf","rgdfg","sdfs"]),
    ("Bob", ["1","2","3"]),
    ("Ace", ["4","5","6"])

]

columns = ["Name", "Sports"]

df = spark.createDataFrame(data, columns)

display(df)

In [0]:
df_new = df.select("Name",explode("Sports").alias("new_col"))
display(df_new)

In [0]:
from pyspark.sql.functions import *


data = [
    ("Kolkata", "","WB"),
    ("","GuruGram",None),
    (None,"","Bangaluru")

]

columns = ["c1", "c2","c3"]

df = spark.createDataFrame(data, columns)

display(df)

In [0]:
df_new = df.withColumn("city4",coalesce(when(col("c1")== "",None).otherwise(col("c1")),
                                        when(col("c2")== "",None).otherwise(col("c2")),
                                        when(col("c3")== "",None).otherwise(col("c3"))))
display(df_new)

In [0]:
from pyspark.sql.functions import *

# Employee DataFrame
emp_data = [
    [100, 'kiran', 100, 1, '01/04/23', 5000],
    [200, 'joanne', 100, 1, '01/04/23', 4000],
    [200, 'joanne', 100, 1, '13/04/23', 4500],
    [200, 'joanne', 100, 1, '14/04/23', 4020]
]

emp_df = pd.DataFrame(emp_data, columns=[
    'EmpId', 'EmpName', 'MgrId', 'DeptId', 'SalaryDate', 'Salary'
])




# Department DataFrame
dept_data = [
    [1, 'IT'],
    [2, 'HR']
]

dept_df = pd.DataFrame(dept_data, columns=[
    'DeptId', 'DeptName'
])

display(emp_df)
display(dept_df)

In [0]:
from pyspark.sql import SparkSession

# Create Spark session
spark = SparkSession.builder.appName("CreateDataFrame").getOrCreate()

# Data from the image
data = [
    (1, "a"),
    (2, "b"),
    (3, "c"),
    (4, "d"),
    (5, "e")
]

# Define columns
columns = ["ID", "Name"]

# Create DataFrame
df = spark.createDataFrame(data, columns)

# Show DataFrame
df.show()


In [0]:
df_